# Full Model Evaluation — EXACT Logic QA

Notebook này chạy full evaluation trên test set và in đầy đủ thống kê:

- Accuracy, macro/micro/weighted F1
- Precision / recall / F1 từng label
- Confusion matrix
- MC vs Yes/No/Unknown split
- Invalid / UNPARSEABLE rate
- Latency / tokens / length statistics
- Error cases để audit

Default input paths theo request:

```text
DATA_PATH = /kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json
TEST_PATH = /kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json
```

Nếu `generated_v4style_300.json` có gold labels, notebook sẽ tính metrics. Nếu không có gold, notebook vẫn in distribution + invalid rate + lưu predictions.


In [1]:
# ============================================================
# Config
# ============================================================
from pathlib import Path
import os, re, json, time, math, random, traceback
from collections import Counter, defaultdict

DATA_PATH = Path('/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json')
TEST_PATH = Path('/kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json')

# Set manually if auto-detection fails.
BASE_MODEL_PATH = None   # e.g. '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1'
ADAPTER_PATH = None      # e.g. '/kaggle/input/your-lora-adapter/checkpoint-xxx', or None for base model only

# Inference controls
MAX_ROWS = None          # set small int for smoke test, None for all rows
MAX_NEW_TOKENS = 384
TEMPERATURE = 0.0       # deterministic by default
TOP_P = 0.9
DO_SAMPLE = TEMPERATURE > 0
BATCH_SIZE = 1           # keep 1 for robust generation with varied prompt lengths
SEED = 42

# Apply only generic safe parser normalization. Do NOT apply v34/v34.2 validation-tuned flips to a new test set by default.
APPLY_VAL_TUNED_POSTPROCESS = False

# Output files
OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
PRED_PATH = OUT_DIR / 'full_model_eval_preds.json'
SUMMARY_PATH = OUT_DIR / 'full_model_eval_summary.json'
CM_CSV_PATH = OUT_DIR / 'full_model_eval_confusion_matrix.csv'
PER_LABEL_CSV_PATH = OUT_DIR / 'full_model_eval_per_label_metrics.csv'
ERROR_CASES_PATH = OUT_DIR / 'full_model_eval_error_cases.json'

random.seed(SEED)
try:
    import numpy as np
    np.random.seed(SEED)
except Exception:
    pass

print('DATA_PATH:', DATA_PATH, 'exists=', DATA_PATH.exists())
print('TEST_PATH:', TEST_PATH, 'exists=', TEST_PATH.exists())
print('OUT_DIR:', OUT_DIR)


DATA_PATH: /kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json exists= True
TEST_PATH: /kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json exists= True
OUT_DIR: /kaggle/working


In [2]:
# ============================================================
# Utility: load JSON and flatten common schemas
# ============================================================
def load_json(path):
    path = Path(path)
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def extract_records(raw):
    if isinstance(raw, list):
        return raw
    if isinstance(raw, dict):
        for k in ['data', 'records', 'samples', 'items', 'examples', 'queries', 'test']:
            if isinstance(raw.get(k), list):
                return raw[k]
        # dict keyed by ids
        if all(isinstance(v, dict) for v in raw.values()):
            out = []
            for k, v in raw.items():
                vv = dict(v)
                vv.setdefault('id', k)
                out.append(vv)
            return out
    raise TypeError(f'Unsupported JSON schema: {type(raw).__name__}')

raw_data = load_json(DATA_PATH) if DATA_PATH.exists() else None
raw_test = load_json(TEST_PATH)

data_rows = extract_records(raw_data) if raw_data is not None else []
test_rows = extract_records(raw_test)
if MAX_ROWS:
    test_rows = test_rows[:MAX_ROWS]

print('Loaded data_rows:', len(data_rows))
print('Loaded test_rows:', len(test_rows))
if test_rows:
    print('First test keys:', sorted(test_rows[0].keys())[:80])
    print(json.dumps(test_rows[0], ensure_ascii=False)[:2000])


Loaded data_rows: 399
Loaded test_rows: 300
First test keys: ['answers', 'explanation', 'idx', 'premises-FOL', 'premises-NL', 'questions']
{"idx": [[1, 2], [3, 4]], "premises-FOL": ["∀x (EnrollsInCourse(x) → AttendsAllLectures(x))", "∃x (AttendsAllLectures(x))", "∀x (SubmitsAllAssignments(x) → AchievesHighGPA(x))", "∀x (AchievesHighGPA(x))", "∃x (ReceivesScholarship(x))", "∀x (ReadsSupplementaryMaterial(x) → TakesTutoringSession(x))", "∀x (CompletesInternship(x))", "∃x (AttendsPeerReview(x))", "∃x (TakesTutoringSession(x))"], "premises-NL": ["If a student enrolls in the course, then the student attends all lectures.", "At least one student attends all lectures.", "If a student submits all assignments, then the student achieves a high GPA.", "Every student achieves a high GPA.", "At least one student receives a scholarship.", "If a student reads supplementary material, then the student takes a tutoring session.", "Every student completes an internship.", "At least one student attends a 

In [3]:
# ============================================================
# Schema helpers: fields, labels, prompts
# ============================================================
LABELS_ALL = ['A','B','C','D','Yes','No','Unknown']
YNU_LABELS = ['Yes','No','Unknown']
MC_LABELS = ['A','B','C','D']

FINAL_RE = re.compile(r'(?:Final\s*Answer|Answer|Đáp\s*án|Label)\s*[:\-]?\s*(Yes|No|Unknown|[ABCD])\b', re.I)
LOOSE_LABEL_RE = re.compile(r'\b(Yes|No|Unknown|[ABCD])\b', re.I)

def normalize_label(x):
    if x is None:
        return None
    s = str(x).strip()
    if not s:
        return None
    up = s.upper()
    if up in MC_LABELS:
        return up
    low = s.lower()
    if low in ['yes', 'true', 'correct']:
        return 'Yes'
    if low in ['no', 'false', 'incorrect']:
        return 'No'
    if low in ['unknown', 'unk', 'cannot determine', 'cannot be determined', 'not enough information', 'insufficient information']:
        return 'Unknown'
    return None

def parse_answer(text, is_mc_hint=None):
    if text is None:
        return 'UNPARSEABLE'
    s = str(text).strip()
    if not s:
        return 'UNPARSEABLE'
    matches = FINAL_RE.findall(s)
    if matches:
        lab = normalize_label(matches[-1])
        if lab:
            return lab
    # Try JSON-ish field
    for pat in [r'"final_answer"\s*:\s*"?([^",}\n]+)', r'"answer"\s*:\s*"?([^",}\n]+)', r'final_answer\s*=\s*([^,}\n]+)']:
        m = re.search(pat, s, flags=re.I)
        if m:
            lab = normalize_label(m.group(1))
            if lab:
                return lab
    # Last line loose parse
    tail = '\n'.join(s.splitlines()[-5:])
    loose = LOOSE_LABEL_RE.findall(tail)
    if loose:
        lab = normalize_label(loose[-1])
        if lab:
            return lab
    return 'UNPARSEABLE'

def first_present(row, keys, default=''):
    for k in keys:
        if k in row and row[k] is not None and row[k] != '':
            return row[k]
    return default

def get_gold(row):
    for k in ['gold', 'answer', 'label', 'target', 'correct_answer', 'final_answer', 'gt', 'y']:
        if k in row:
            lab = normalize_label(row[k])
            if lab:
                return lab
    return None

def infer_is_mc(row):
    # explicit flag
    for k in ['is_mc','is_multiple_choice','multiple_choice','q_type','question_type','type']:
        if k in row:
            v = row[k]
            if isinstance(v, bool):
                return v
            sv = str(v).lower()
            if any(t in sv for t in ['mc','multiple','choice','abcd']): return True
            if any(t in sv for t in ['ynu','yes/no','yes_no','boolean']): return False
    q = str(first_present(row, ['question','query','prompt','input'], ''))
    opts = []
    for k in ['options','choices']:
        if isinstance(row.get(k), (list, dict)):
            return True
    if re.search(r'\bA\s*[\.)]', q) and re.search(r'\bB\s*[\.)]', q):
        return True
    gold = get_gold(row)
    if gold in MC_LABELS: return True
    if gold in YNU_LABELS: return False
    return False

def format_premises(row):
    premises = first_present(row, ['premises','context','facts','rules','passage','story','background'], None)
    if premises is None:
        return ''
    if isinstance(premises, list):
        lines = []
        for i, p in enumerate(premises, 1):
            if isinstance(p, dict):
                txt = first_present(p, ['text','premise','content','sentence'], json.dumps(p, ensure_ascii=False))
            else:
                txt = str(p)
            lines.append(f'{i}. {txt}')
        return '\n'.join(lines)
    if isinstance(premises, dict):
        return '\n'.join(f'{k}. {v}' for k, v in premises.items())
    return str(premises)

def format_options(row):
    opts = first_present(row, ['options','choices','answers'], None)
    if opts is None:
        q = str(first_present(row, ['question','query','prompt','input'], ''))
        # options may already be embedded in question
        return ''
    if isinstance(opts, dict):
        return '\n'.join(f'{k}. {v}' for k, v in opts.items())
    if isinstance(opts, list):
        letters = 'ABCD'
        return '\n'.join(f'{letters[i]}. {v}' for i, v in enumerate(opts[:4]))
    return str(opts)

def build_prompt(row):
    question = str(first_present(row, ['question','query','prompt','input'], ''))
    premises = format_premises(row)
    options = format_options(row)
    is_mc = infer_is_mc(row)
    label_space = 'A, B, C, or D' if is_mc else 'Yes, No, or Unknown'
    parts = []
    parts.append('You are solving a logic-based educational QA problem. Use only the given premises. Do not use outside knowledge.')
    if premises:
        parts.append('Premises:\n' + premises)
    parts.append('Question:\n' + question)
    if options and options not in question:
        parts.append('Options:\n' + options)
    parts.append(f'Reason step by step briefly, cite supporting premises if useful, and end with exactly one line: Final Answer: <{label_space}>')
    return '\n\n'.join(parts)

# Align/fill idx/gold/is_mc metadata
for i, r in enumerate(test_rows):
    r.setdefault('idx', i)

print('Gold availability:', sum(get_gold(r) is not None for r in test_rows), '/', len(test_rows))
print('MC count:', sum(infer_is_mc(r) for r in test_rows), 'YNU count:', sum(not infer_is_mc(r) for r in test_rows))
print('Gold distribution:', Counter(get_gold(r) or 'NO_GOLD' for r in test_rows))
print('Prompt preview:\n', build_prompt(test_rows[0])[:2500] if test_rows else '')


Gold availability: 0 / 300
MC count: 0 YNU count: 300
Gold distribution: Counter({'NO_GOLD': 300})
Prompt preview:
 You are solving a logic-based educational QA problem. Use only the given premises. Do not use outside knowledge.

Question:


Options:
A. Unknown
B. Unknown

Reason step by step briefly, cite supporting premises if useful, and end with exactly one line: Final Answer: <Yes, No, or Unknown>


In [4]:
# ============================================================
# Model discovery and loading
# ============================================================
def find_model_paths():
    candidates = []
    roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path('/mnt/data')]
    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob('config.json'):
            # Prefer Qwen-ish causal LM folders but keep all candidates
            parent = p.parent
            name = str(parent).lower()
            score = 0
            if 'qwen' in name: score += 10
            if '8b' in name or '7b' in name: score += 3
            if 'adapter' in name or 'lora' in name: score -= 3
            candidates.append((score, str(parent)))
    candidates = sorted(set(candidates), reverse=True)
    return [p for _, p in candidates]

def find_adapter_paths():
    candidates = []
    roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path('/mnt/data')]
    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob('adapter_config.json'):
            candidates.append(str(p.parent))
    return sorted(set(candidates))

model_candidates = find_model_paths()
adapter_candidates = find_adapter_paths()
print('Model candidates (top 10):')
for p in model_candidates[:10]: print(' ', p)
print('Adapter candidates (top 10):')
for p in adapter_candidates[:10]: print(' ', p)

if BASE_MODEL_PATH is None:
    BASE_MODEL_PATH = model_candidates[0] if model_candidates else None
if ADAPTER_PATH is None:
    ADAPTER_PATH = adapter_candidates[0] if adapter_candidates else None

print('Using BASE_MODEL_PATH:', BASE_MODEL_PATH)
print('Using ADAPTER_PATH:', ADAPTER_PATH)


Model candidates (top 10):
  /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1
Adapter candidates (top 10):
  /kaggle/input/datasets/yixuanisthebest/v2333333
Using BASE_MODEL_PATH: /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1
Using ADAPTER_PATH: /kaggle/input/datasets/yixuanisthebest/v2333333


In [5]:
# ============================================================
# Load model. If unavailable, notebook can evaluate existing pred fields only.
# ============================================================
model = None
tok = None
load_error = None
load_mode = None

try:
    if BASE_MODEL_PATH is None:
        raise FileNotFoundError('No base model found. Set BASE_MODEL_PATH manually.')
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    try:
        from peft import PeftModel
    except Exception:
        PeftModel = None

    tok_path = ADAPTER_PATH or BASE_MODEL_PATH
    tok = AutoTokenizer.from_pretrained(tok_path, trust_remote_code=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

    # bnb 4-bit only if available; otherwise normal sharded dtype.
    quant_config = None
    try:
        import bitsandbytes as bnb  # noqa
        from transformers import BitsAndBytesConfig
        quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4')
        load_mode = 'bnb4'
    except Exception:
        load_mode = 'sharded_dtype'

    kw = dict(trust_remote_code=True, device_map='auto')
    if quant_config is not None:
        kw['quantization_config'] = quant_config
    else:
        kw['dtype'] = dtype

    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_PATH, **kw)
    if ADAPTER_PATH:
        if PeftModel is None:
            raise ImportError('peft not available but ADAPTER_PATH was set')
        model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    model.eval()
    print('MODEL LOADED:', load_mode)
except Exception as e:
    load_error = traceback.format_exc()
    print('MODEL LOAD FAILED. Will try to evaluate existing prediction fields if present.')
    print(load_error[:4000])


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

MODEL LOADED: sharded_dtype


In [6]:
# ============================================================
# Inference
# ============================================================
def existing_prediction(row):
    for k in ['pred','prediction','model_pred','output_answer','generated_answer']:
        if k in row:
            lab = normalize_label(row[k])
            if lab:
                return lab, str(row[k]), f'existing:{k}'
    # If row has raw generation, parse it
    for k in ['output','completion','response','generation','text','explanation']:
        if k in row:
            lab = parse_answer(row[k], infer_is_mc(row))
            if lab != 'UNPARSEABLE':
                return lab, str(row[k]), f'parse_existing:{k}'
    return None, None, None

def generate_one(prompt):
    assert model is not None and tok is not None
    import torch
    messages = [{'role':'user','content':prompt}]
    try:
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        text = prompt
    inp = tok(text, return_tensors='pt', truncation=True, max_length=4096)
    # Move tensor dict to model device safely. For device_map='auto', first parameter device is OK for input_ids.
    device = next(model.parameters()).device
    inp = {k: v.to(device) for k, v in inp.items()}
    gen_kwargs = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=DO_SAMPLE, pad_token_id=tok.eos_token_id)
    if DO_SAMPLE:
        gen_kwargs.update(temperature=TEMPERATURE, top_p=TOP_P)
    with torch.no_grad():
        out = model.generate(**inp, **gen_kwargs)
    new_tokens = out[0][inp['input_ids'].shape[-1]:]
    return tok.decode(new_tokens, skip_special_tokens=True)

pred_rows = []
t0_all = time.time()

for n, row in enumerate(test_rows):
    idx = row.get('idx', n)
    gold = get_gold(row)
    is_mc = infer_is_mc(row)
    prompt = build_prompt(row)
    t0 = time.time()
    raw_output = None
    source = None
    try:
        if model is not None:
            raw_output = generate_one(prompt)
            pred = parse_answer(raw_output, is_mc)
            source = 'generated'
        else:
            pred, raw_output, source = existing_prediction(row)
            if pred is None:
                pred, raw_output, source = 'UNPARSEABLE', '', 'none'
    except Exception as e:
        pred = 'UNPARSEABLE'
        raw_output = traceback.format_exc()
        source = 'generation_error'
    dt = time.time() - t0
    pred_rows.append({
        'idx': idx,
        'gold': gold,
        'pred': pred,
        'is_mc': bool(is_mc),
        'source': source,
        'latency_sec': dt,
        'question': first_present(row, ['question','query','prompt','input'], ''),
        'prompt': prompt,
        'raw_output': raw_output,
    })
    if (n+1) % 10 == 0 or n == 0:
        print(f'[{n+1}/{len(test_rows)}] pred={pred} gold={gold} source={source} avg_latency={sum(r["latency_sec"] for r in pred_rows)/(n+1):.2f}s')

print('Total inference/eval time sec:', round(time.time() - t0_all, 2))
with open(PRED_PATH, 'w', encoding='utf-8') as f:
    json.dump(pred_rows, f, ensure_ascii=False, indent=2)
print('Saved preds:', PRED_PATH)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[1/300] pred=Unknown gold=None source=generated avg_latency=7.69s
[10/300] pred=Yes gold=None source=generated avg_latency=9.11s
[20/300] pred=Yes gold=None source=generated avg_latency=11.07s
[30/300] pred=Yes gold=None source=generated avg_latency=11.77s
[40/300] pred=Unknown gold=None source=generated avg_latency=11.60s
[50/300] pred=Unknown gold=None source=generated avg_latency=11.26s
[60/300] pred=Unknown gold=None source=generated avg_latency=11.76s
[70/300] pred=Unknown gold=None source=generated avg_latency=12.59s
[80/300] pred=Yes gold=None source=generated avg_latency=12.27s
[90/300] pred=Unknown gold=None source=generated avg_latency=12.76s
[100/300] pred=Yes gold=None source=generated avg_latency=12.76s
[110/300] pred=Yes gold=None source=generated avg_latency=12.83s
[120/300] pred=Unknown gold=None source=generated avg_latency=12.49s
[130/300] pred=Unknown gold=None source=generated avg_latency=12.19s
[140/300] pred=Unknown gold=None source=generated avg_latency=12.57s
[1

In [7]:
# ============================================================
# Metrics: accuracy, precision, recall, F1, confusion matrix, splits
# ============================================================
def prf_counts(rows, labels):
    res = {}
    for lab in labels:
        tp = sum(1 for r in rows if r.get('gold') == lab and r.get('pred') == lab)
        fp = sum(1 for r in rows if r.get('gold') != lab and r.get('pred') == lab)
        fn = sum(1 for r in rows if r.get('gold') == lab and r.get('pred') != lab)
        support = sum(1 for r in rows if r.get('gold') == lab)
        pred_count = sum(1 for r in rows if r.get('pred') == lab)
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
        res[lab] = dict(tp=tp, fp=fp, fn=fn, precision=prec, recall=rec, f1=f1, support=support, pred_count=pred_count)
    return res

def compute_metrics(rows, label_order=None):
    eval_rows = [r for r in rows if r.get('gold') is not None]
    if label_order is None:
        labs = []
        for lab in LABELS_ALL:
            if any(r.get('gold') == lab or r.get('pred') == lab for r in eval_rows):
                labs.append(lab)
        for lab in sorted(set((r.get('gold') for r in eval_rows)) | set((r.get('pred') for r in eval_rows))):
            if lab and lab not in labs:
                labs.append(lab)
    else:
        labs = list(label_order)
    n = len(eval_rows)
    correct = sum(1 for r in eval_rows if r.get('gold') == r.get('pred'))
    per = prf_counts(eval_rows, labs)
    macro = sum(per[l]['f1'] for l in labs)/len(labs) if labs else None
    weighted = sum(per[l]['f1'] * per[l]['support'] for l in labs)/sum(per[l]['support'] for l in labs) if labs and sum(per[l]['support'] for l in labs) else None
    micro_tp = sum(per[l]['tp'] for l in labs)
    # single-label micro-F1 equals accuracy over included labels, but keep explicit
    micro = correct/n if n else None
    cm = {g:{p:0 for p in labs+['UNPARSEABLE','OTHER']} for g in labs}
    for r in eval_rows:
        g, p = r.get('gold'), r.get('pred')
        if g not in cm:
            continue
        pp = p if p in cm[g] else 'OTHER'
        cm[g][pp] += 1
    return dict(n=n, correct=correct, acc=correct/n if n else None, micro_f1=micro, macro_f1=macro, weighted_f1=weighted, labels=labs, per_label=per, confusion_matrix=cm)

def split_metrics(rows):
    out = {}
    eval_rows = [r for r in rows if r.get('gold') is not None]
    out['all'] = compute_metrics(eval_rows)
    out['mc'] = compute_metrics([r for r in eval_rows if r.get('is_mc')], MC_LABELS)
    out['ynu'] = compute_metrics([r for r in eval_rows if not r.get('is_mc')], YNU_LABELS)
    return out

metrics = split_metrics(pred_rows)

# Distribution/quality stats even without gold
pred_dist = Counter(r['pred'] for r in pred_rows)
gold_dist = Counter(r['gold'] if r['gold'] is not None else 'NO_GOLD' for r in pred_rows)
invalid_count = sum(1 for r in pred_rows if r['pred'] == 'UNPARSEABLE')
latencies = [r['latency_sec'] for r in pred_rows]

summary = {
    'version': 'full_model_eval_metrics',
    'data_path': str(DATA_PATH),
    'test_path': str(TEST_PATH),
    'base_model_path': BASE_MODEL_PATH,
    'adapter_path': ADAPTER_PATH,
    'load_mode': load_mode,
    'load_error': load_error,
    'n_rows': len(pred_rows),
    'gold_available': sum(r.get('gold') is not None for r in pred_rows),
    'pred_distribution': dict(pred_dist),
    'gold_distribution': dict(gold_dist),
    'invalid_unparseable_count': invalid_count,
    'invalid_unparseable_rate': invalid_count / len(pred_rows) if pred_rows else None,
    'latency_sec': {
        'total': sum(latencies),
        'mean': sum(latencies)/len(latencies) if latencies else None,
        'min': min(latencies) if latencies else None,
        'max': max(latencies) if latencies else None,
    },
    'metrics': metrics,
}

# Error cases
error_cases = [r for r in pred_rows if r.get('gold') is not None and r.get('gold') != r.get('pred')]
with open(SUMMARY_PATH, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
with open(ERROR_CASES_PATH, 'w', encoding='utf-8') as f:
    json.dump(error_cases, f, ensure_ascii=False, indent=2)

print('Saved summary:', SUMMARY_PATH)
print('Saved error cases:', ERROR_CASES_PATH)


Saved summary: /kaggle/working/full_model_eval_summary.json
Saved error cases: /kaggle/working/full_model_eval_error_cases.json


In [8]:
# ============================================================
# Pretty print metrics and save CSV tables
# ============================================================
import pandas as pd

print('\n================ OVERALL ================')
print('Rows:', summary['n_rows'])
print('Gold available:', summary['gold_available'])
print('Prediction distribution:', summary['pred_distribution'])
print('Gold distribution:', summary['gold_distribution'])
print('UNPARSEABLE:', summary['invalid_unparseable_count'], f"({summary['invalid_unparseable_rate']:.2%})")
print('Latency mean/sec:', summary['latency_sec']['mean'])

for split in ['all','mc','ynu']:
    m = metrics[split]
    print(f'\n================ {split.upper()} METRICS ================')
    print('n:', m['n'])
    print('correct:', m['correct'])
    print('accuracy:', m['acc'])
    print('micro_f1:', m['micro_f1'])
    print('macro_f1:', m['macro_f1'])
    print('weighted_f1:', m['weighted_f1'])
    df = pd.DataFrame(m['per_label']).T
    if not df.empty:
        display(df[['precision','recall','f1','support','pred_count','tp','fp','fn']])

# Confusion matrix for all labels
m_all = metrics['all']
cm_df = pd.DataFrame(m_all['confusion_matrix']).T.fillna(0).astype(int)
print('\n================ CONFUSION MATRIX: rows=gold, cols=pred ================')
display(cm_df)
cm_df.to_csv(CM_CSV_PATH)

per_rows = []
for split in ['all','mc','ynu']:
    for lab, vals in metrics[split]['per_label'].items():
        row = {'split': split, 'label': lab}
        row.update(vals)
        per_rows.append(row)
per_df = pd.DataFrame(per_rows)
per_df.to_csv(PER_LABEL_CSV_PATH, index=False)
print('Saved confusion matrix:', CM_CSV_PATH)
print('Saved per-label metrics:', PER_LABEL_CSV_PATH)

# Show top error cases compactly
print('\n================ ERROR CASES PREVIEW ================')
err_preview = []
for r in error_cases[:50]:
    err_preview.append({
        'idx': r['idx'], 'is_mc': r['is_mc'], 'gold': r['gold'], 'pred': r['pred'],
        'question': str(r['question'])[:180],
        'raw_output_tail': str(r['raw_output'])[-300:]
    })
if err_preview:
    display(pd.DataFrame(err_preview))
else:
    print('No errors, or gold labels unavailable.')



================ OVERALL ================
Rows: 300
Gold available: 0
Prediction distribution: {'Unknown': 145, 'Yes': 130, 'A': 25}
Gold distribution: {'NO_GOLD': 300}
UNPARSEABLE: 0 (0.00%)
Latency mean/sec: 12.858907779057821

================ ALL METRICS ================
n: 0
correct: 0
accuracy: None
micro_f1: None
macro_f1: None
weighted_f1: None

================ MC METRICS ================
n: 0
correct: 0
accuracy: None
micro_f1: None
macro_f1: 0.0
weighted_f1: None


,precision,recall,f1,support,pred_count,tp,fp,fn
A,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
B,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
C,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
D,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



================ YNU METRICS ================
n: 0
correct: 0
accuracy: None
micro_f1: None
macro_f1: 0.0
weighted_f1: None


,precision,recall,f1,support,pred_count,tp,fp,fn
Yes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
No,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Unknown,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



================ CONFUSION MATRIX: rows=gold, cols=pred ================


""


Saved confusion matrix: /kaggle/working/full_model_eval_confusion_matrix.csv
Saved per-label metrics: /kaggle/working/full_model_eval_per_label_metrics.csv

================ ERROR CASES PREVIEW ================
No errors, or gold labels unavailable.


In [9]:
# ============================================================
# Additional diagnostic tables
# ============================================================
import pandas as pd

df = pd.DataFrame(pred_rows)
print('Rows by source:')
display(df['source'].value_counts(dropna=False).to_frame('count'))
print('\nRows by pred/is_mc:')
display(pd.crosstab(df['pred'], df['is_mc'], margins=True))
if df['gold'].notna().any():
    print('\nAccuracy by is_mc:')
    tmp = df[df['gold'].notna()].copy()
    tmp['correct'] = tmp['gold'] == tmp['pred']
    display(tmp.groupby('is_mc').agg(n=('idx','count'), acc=('correct','mean'), unparseable=('pred', lambda s: (s=='UNPARSEABLE').mean())))
    print('\nError type counts gold->pred:')
    errs = tmp[~tmp['correct']]
    display(errs.assign(pair=errs['gold'].astype(str)+' -> '+errs['pred'].astype(str))['pair'].value_counts().to_frame('count'))

print('\nOutput files:')
for p in [PRED_PATH, SUMMARY_PATH, CM_CSV_PATH, PER_LABEL_CSV_PATH, ERROR_CASES_PATH]:
    print(p)


Rows by source:


,count
source,
generated,300



Rows by pred/is_mc:


is_mc,False,All
pred,,
A,25,25
Unknown,145,145
Yes,130,130
All,300,300



Output files:
/kaggle/working/full_model_eval_preds.json
/kaggle/working/full_model_eval_summary.json
/kaggle/working/full_model_eval_confusion_matrix.csv
/kaggle/working/full_model_eval_per_label_metrics.csv
/kaggle/working/full_model_eval_error_cases.json
